# Exploration initiale

Objectif du projet : trouver les principaux postes de dépense par service et par région, et repérer les ressources qui coûtent cher mais qui sont peu utilisées (gaspillage cloud potentiel).

Cette étape : premier coup d'œil au dataset. On regarde la taille, les types de colonnes, les valeurs manquantes, et un résumé des colonnes. On ne modifie rien ici.

Source : `data/raw/gcp_final_approved_dataset.csv` (1000 lignes, 15 colonnes)

In [27]:
import pandas as pd

df = pd.read_csv("../data/raw/gcp_final_approved_dataset.csv")
df.shape

(1000, 15)

In [31]:
df.head()

,Resource ID,Service Name,Usage Quantity,Usage Unit,Region/Zone,CPU Utilization (%),Memory Utilization (%),Network Inbound Data (Bytes),Network Outbound Data (Bytes),Usage Start Date,Usage End Date,Cost per Quantity ($),Unrounded Cost ($),Rounded Cost ($),Total Cost (INR)
0,res-ST6BAJ2N,Cloud Dataproc,954.9843,Requests,europe-north1,92.72,70.42,77097035547,8.268593e+10,01-08-2024 22:24,07-08-2024 06:54,5.24,5004.12,5004,415332
1,res-TYGNR0RV,Pub/Sub,479.6348,GB,europe-west1,37.78,46.80,8239357017,1.135149e+10,23-08-2024 09:18,25-08-2024 07:26,9.82,4710.01,4710,390930
2,res-S3I9C869,BigQuery,114.4129,GB,southamerica-east1,82.54,59.47,23358691883,2.345081e+10,18-07-2024 11:13,22-07-2024 00:45,7.37,843.22,843,69969
3,res-1RY9BZ6G,Cloud Endpoints,413.1930,GB,us-central1,86.68,97.63,53404385778,6.078019e+10,20-07-2024 00:15,21-07-2024 10:16,4.41,1822.18,1822,151226
4,res-S3HQYIZ2,Cloud Spanner,335.9892,GB,asia-southeast1,82.95,21.58,10795270461,1.416138e+10,21-08-2024 21:25,25-08-2024 00:21,6.09,2046.17,2046,169818


Les Date sont stocké en str, à convertir en datetime

In [30]:
df.dtypes

Resource ID                          str
Service Name                         str
Usage Quantity                   float64
Usage Unit                           str
Region/Zone                          str
CPU Utilization (%)              float64
Memory Utilization (%)           float64
Network Inbound Data (Bytes)       int64
Network Outbound Data (Bytes)    float64
Usage Start Date                     str
Usage End Date                       str
Cost per Quantity ($)            float64
Unrounded Cost ($)               float64
Rounded Cost ($)                   int64
Total Cost (INR)                   int64
dtype: object

In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Resource ID                    1000 non-null   str    
 1   Service Name                   1000 non-null   str    
 2   Usage Quantity                 1000 non-null   float64
 3   Usage Unit                     1000 non-null   str    
 4   Region/Zone                    1000 non-null   str    
 5   CPU Utilization (%)            1000 non-null   float64
 6   Memory Utilization (%)         1000 non-null   float64
 7   Network Inbound Data (Bytes)   1000 non-null   int64  
 8   Network Outbound Data (Bytes)  1000 non-null   float64
 9   Usage Start Date               1000 non-null   str    
 10  Usage End Date                 1000 non-null   str    
 11  Cost per Quantity ($)          1000 non-null   float64
 12  Unrounded Cost ($)             1000 non-null   float64
 13  

In [29]:
df.describe()

,Usage Quantity,CPU Utilization (%),Memory Utilization (%),Network Inbound Data (Bytes),Network Outbound Data (Bytes),Cost per Quantity ($),Unrounded Cost ($),Rounded Cost ($),Total Cost (INR)
count,1000.000000,1000.000000,1000.000000,1.000000e+03,1.000000e+03,1000.000000,1000.000000,1000.000000,1000.000000
mean,383.343530,50.943680,51.629960,5.152107e+10,5.648232e+10,5.515040,2135.215130,2135.215000,177222.845000
std,299.815662,28.824742,28.367572,2.826950e+10,2.837284e+10,2.539737,2080.530698,2080.553012,172685.899993
min,10.540200,0.050000,0.060000,1.003429e+09,1.207921e+09,1.000000,13.910000,14.000000,1162.000000
25%,106.818150,26.740000,27.935000,2.747956e+10,3.347126e+10,3.260000,500.490000,500.500000,41541.500000
50%,336.721800,52.030000,51.635000,5.261220e+10,5.781499e+10,5.560000,1251.155000,1251.000000,103833.000000
75%,616.048825,75.647500,75.900000,7.577308e+10,8.077774e+10,7.702500,3257.315000,3257.500000,270372.500000
max,998.005500,99.970000,99.810000,9.983793e+10,1.087820e+11,9.990000,9003.650000,9004.000000,747332.000000


## Colonnes catégorielles

On regarde les colonnes texte qui vont servir à regrouper l'analyse : Service Name et Region/Zone.

- `nunique()` : nombre de valeurs différentes
- `value_counts()` : nombre de lignes pour chaque valeur (trié du plus fréquent au moins fréquent)

In [34]:
df["Service Name"].nunique()

25

In [28]:
df["Region/Zone"].nunique()

12

In [24]:
df["Service Name"].value_counts()

Service Name
Cloud Run               55
Pub/Sub                 51
Cloud CDN               51
Cloud Build             48
Compute Engine          47
Cloud Storage           45
Firestore               44
Cloud Interconnect      42
Cloud Endpoints         41
AI Platform             40
Kubernetes Engine       40
BigQuery                39
Cloud Armor             39
Cloud Memorystore       39
Cloud Dataproc          38
Cloud VPC               38
Cloud Functions         38
Cloud VPN               37
Dataflow                36
Cloud Spanner           35
Cloud NAT               35
Cloud SQL               35
Cloud Load Balancing    32
Container Registry      30
Cloud Data Fusion       25
Name: count, dtype: int64

In [26]:
df["Region/Zone"].value_counts()

Region/Zone
asia-east1                 103
europe-west1                99
asia-southeast1             91
southamerica-east1          89
europe-north1               83
northamerica-northeast1     82
us-east1                    82
us-central1                 81
europe-west3                77
asia-northeast1             74
australia-southeast1        71
us-west1                    68
Name: count, dtype: int64

## Synthèse de l'exploration

- Le dataset a 1000 lignes et 15 colonnes.
- Aucune valeur manquante (toutes les colonnes sont à 1000 non-null).
- Les colonnes `Usage Start Date` et `Usage End Date` sont stockées en texte (str), il faut les convertir en datetime à l'étape 2.
- Le coût est présent 3 fois : `Unrounded Cost ($)`, `Rounded Cost ($)` et `Total Cost (INR)`. Le INR est juste le montant en dollars converti en roupies (environ x83). On garde `Unrounded Cost ($)` pour l'analyse.
- La distribution des coûts est asymétrique : médiane 1251 mais moyenne 2135 , donc quelques ressources très chères tirent la moyenne vers le haut.
- Il y a 25 services et 12 régions, bien répartis (au moins 25 lignes par groupe), donc les regroupements par service / région seront fiables.
- La colonne `Usage Quantity` mélange des unités différentes (Requests, GB, etc.), on ne peut pas l'additionner telle quelle.